In [1]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

False

In [2]:
# Create an API Clent
from anthropic import Anthropic

client = Anthropic()
model = "openrouter/free"

In [3]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
#########################################
# Request without temperature parameter #
#########################################

messages = []

add_user_message(
    messages, "Write a Python function that checks a string for duplicate characters"
)
answer = chat(messages, system="You are a Python engineer who writes very concise code")

answer

In [12]:
######################################
# Request with temperature parameter #
######################################

messages = []

add_user_message(
    messages, "Generate a one sentence movie idea"
)
answer = chat(messages, temperature=1.0)

answer

'A disillusioned librarian discovers a hidden code within antique books that unlocks forgotten memories and reveals a conspiracy to rewrite history. \n'

In [13]:
######################
# Response Streaming #
######################

messages = []

add_user_message(messages, 'Write a 1 sentence description of a fake database')

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)
for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='gen-1775495050-DUQUJ8UdVxyzeE661g75', container=None, content=[], model='minimax/minimax-m2.5-20260211:free', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=None, cache_creation_input_tokens=None, cache_read_input_tokens=None, inference_geo=None, input_tokens=0, output_tokens=0, server_tool_use=None, service_tier=None, speed='standard'), provider='OpenInference'), type='message_start')
RawContentBlockStartEvent(content_block=ThinkingBlock(signature='', thinking='', type='thinking'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=ThinkingDelta(thinking='The', type='thinking_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=ThinkingDelta(thinking=' user', type='thinking_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=ThinkingDelta(thinking=' asks', type='thinking_delta'), index=0, type='content_bloc

In [14]:
###################################
# Response Streaming in fragments #
###################################

messages = []

add_user_message(messages, 'Write a 1 sentence description of a fake database')

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # print(text, end="") Remove # for fragments visualization
        pass

stream.get_final_message() # Ensamble of all messages in one

ParsedMessage(id='gen-1775495084-yDXohfTgMYBkroUgwEYt', container=None, content=[ThinkingBlock(signature='', thinking='We need to produce one sentence describing a fake database.', type='thinking'), ParsedTextBlock(citations=None, text='A whimsical, query‑powered “Chronicle of Unfinished Dreams” database that stores every half‑remembered idea, abandoned plot twist, and stray inspiration—indexed by mood, color, and the time of day they first appeared.', type='text', parsed_output=None), RedactedThinkingBlock(data='openrouter.reasoning:eyJ0ZXh0IjoiV2UgbmVlZCB0byBwcm9kdWNlIG9uZSBzZW50ZW5jZSBkZXNjcmliaW5nIGEgZmFrZSBkYXRhYmFzZS4iLCJ0eXBlIjoicmVhc29uaW5nLnRleHQifQ==', type='redacted_thinking')], model='openai/gpt-oss-120b:free', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=None, cache_creation_input_tokens=None, cache_read_input_tokens=64, inference_geo=None, input_tokens=13, output_tokens=68, server_tool_use=None